# Ant Colony Optimisation: a six-stop delivery route

This notebook works through one ACO iteration for a small delivery route. It calculates move probabilities, compares three tours, and updates pheromone using evaporation and deposits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Distance matrix and initial trails

`alpha` weights pheromone, `beta` weights the distance heuristic, `rho` is the evaporation rate, and `Q` sets the deposit amount.

In [1]:
stops = ["Depot", "Market", "Campus", "Station", "Mall", "Park"]
distance = np.array([[0,4,7,9,6,8],[4,0,5,6,3,7],[7,5,0,4,6,3],[9,6,4,0,5,4],[6,3,6,5,0,5],[8,7,3,4,5,0]], dtype=float)
alpha, beta, rho, Q = 1, 2, 0.4, 90
pheromone = np.ones_like(distance)
np.fill_diagonal(pheromone, 0)
print("Distance matrix (rows and columns follow stops list):")
print(distance)
print("\nInitial pheromone:")
print(pheromone)

Distance matrix (rows and columns follow stops list):
[[0,4,7,9,6,8],[4,0,5,6,3,7],[7,5,0,4,6,3],[9,6,4,0,5,4],[6,3,6,5,0,5],[8,7,3,4,5,0]]

Initial pheromone:
[[0,1,1,1,1,1],[1,0,1,1,1,1],[1,1,0,1,1,1],[1,1,1,0,1,1],[1,1,1,1,0,1],[1,1,1,1,1,0]]

## 2. Transition probabilities

For an available next stop, the weight is \(\tau_{ij}^{\alpha}(1/d_{ij})^{\beta}\). Divide each weight by the total for the unvisited choices.

In [1]:
eta = np.zeros_like(distance)
np.divide(1, distance, out=eta, where=distance > 0)
def move_probabilities(current, candidates, trail):
    weights = trail[current, candidates]**alpha * eta[current, candidates]**beta
    return weights / weights.sum()
candidates = [1, 2, 3, 4, 5]
p0 = move_probabilities(0, candidates, pheromone)
for j, p in zip(candidates, p0):
    print(f"{stops[j]:>8}: {p:.4f}")

Possible first moves from Depot
  Market: 0.4508
  Campus: 0.1472
 Station: 0.0890
    Mall: 0.2003
    Park: 0.1127

## 3. Tours selected for this worked iteration

These three tours are fixed so the calculations can be checked. A full ACO run would sample routes according to the probabilities above.

In [1]:
tours = [[0,1,4,3,2,5,0], [1,0,5,2,3,4,1], [2,5,3,4,1,0,2]]
for k, tour in enumerate(tours, 1):
    print(f"Ant {k}: " + " -> ".join(stops[i] for i in tour))

Ant 1: Depot -> Market -> Mall -> Station -> Campus -> Park -> Depot
Ant 2: Market -> Depot -> Park -> Campus -> Station -> Mall -> Market
Ant 3: Campus -> Park -> Station -> Mall -> Market -> Depot -> Campus

## 4. Tour lengths and deposits

Tour length includes the return edge. Each ant deposits \(Q/L_k\) on every edge it uses, so a shorter tour leaves more pheromone.

In [1]:
lengths = []
for k, tour in enumerate(tours, 1):
    costs = distance[tour[:-1], tour[1:]]
    length = costs.sum()
    lengths.append(length)
    print(f"Ant {k}: " + " + ".join(str(int(x)) for x in costs) + f" = {length:.0f}")
lengths = np.array(lengths)
rewards = Q / lengths
print("\nDeposit per used edge:")
for k, reward in enumerate(rewards, 1):
    print(f"Ant {k}: {Q}/{lengths[k-1]:.0f} = {reward:.4f}")

Ant 1: 4 + 3 + 5 + 4 + 3 + 8 = 27
Ant 2: 4 + 8 + 3 + 4 + 5 + 3 = 27
Ant 3: 3 + 4 + 5 + 3 + 4 + 7 = 26

Deposit per used edge:
Ant 1: 90/27 = 3.3333
Ant 2: 90/27 = 3.3333
Ant 3: 90/26 = 3.4615

## 5. Evaporation and pheromone update

First retain \((1-\rho)\tau_{ij}\) of the old trail, then add deposits from every ant. Since routes are undirected, each deposit is recorded in both directions.

In [1]:
deposits = np.zeros_like(distance)
for tour, reward in zip(tours, rewards):
    for i, j in zip(tour[:-1], tour[1:]):
        deposits[i,j] += reward
        deposits[j,i] += reward
evaporated = (1-rho) * pheromone
updated = evaporated + deposits
print("Combined deposits:")
print(deposits)
print("\nUpdated pheromone:")
print(updated)

Combined deposits:
Depot: 0.000  10.128  3.462  0.000  0.000  6.667
Market: 10.128  0.000  0.000  0.000  10.128  0.000
Campus: 3.462  0.000  0.000  6.667  0.000  10.128
Station: 0.000  0.000  6.667  0.000  10.128  3.462
Mall: 0.000  10.128  0.000  10.128  0.000  0.000
Park: 6.667  0.000  10.128  3.462  0.000  0.000

Updated pheromone:
Depot: 0.000  10.728  4.062  0.600  0.600  7.267
Market: 10.728  0.000  0.600  0.600  10.728  0.600
Campus: 4.062  0.600  0.000  7.267  0.600  10.728
Station: 0.600  0.600  7.267  0.000  10.728  4.062
Mall: 0.600  10.728  0.600  10.728  0.000  0.600
Park: 7.267  0.600  10.728  4.062  0.600  0.000

## 6. Change in the next move

The distances stay fixed, but the new pheromone values change the transition probabilities from Depot.

In [1]:
p_before = move_probabilities(0, candidates, pheromone)
p_after = move_probabilities(0, candidates, updated)
for j, x, y in zip(candidates, p_before, p_after):
    print(f"Depot -> {stops[j]}: {x:.4f} -> {y:.4f}")
x = np.arange(len(candidates))
plt.figure(figsize=(8,4))
plt.bar(x-.18, p_before, .36, label="Before update")
plt.bar(x+.18, p_after, .36, label="After update")
plt.xticks(x, [stops[j] for j in candidates])
plt.ylabel("Transition probability")
plt.xlabel("Possible next stop from Depot")
plt.title("Effect of pheromone update")
plt.legend()
plt.tight_layout()
plt.show()

Depot -> Market: 0.4508 -> 0.7525
Depot -> Campus: 0.1472 -> 0.0930
Depot -> Station: 0.0890 -> 0.0083
Depot -> Mall: 0.2003 -> 0.0187
Depot -> Park: 0.1127 -> 0.1274

## Result

The shortest route among these three tours is Ant 3, with total distance **26**. Its edges receive a slightly larger deposit than the tours of length 27. The Depot-to-Market probability rises from about **0.4508** to **0.7525** after the update. More iterations would be needed to explore additional tours.